# CFU prediction using TPM values from bulk RNA-seq

## Data loading

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np

Load TPM data and CFU counts.

In [2]:
from src.tpm_data import get_all_tpm_data

# Paths to Fcnts and CFU data
fcnts_path = "C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab/fcnts_timezero"
cfu_path   = "C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab/all_cfus"

# Get data
data_df = get_all_tpm_data(
    fcnts_path = fcnts_path,
    cfu_path = cfu_path
)

In [3]:
data_df

,SP_0001,SP_0002,SP_0003,SP_0004,SP_0005,SP_0006,SP_0007,SP_0008,SP_0009,SP_0010,...,SP_2239,SP_2240,CFU,drug_id,num_drugs,drug1,drug2,drug1_dose,drug2_dose,timepoint
Labels,,,,,,,,,,,,,,,,,,,,,
NDC1hr-a,91.887204,73.003181,59.434830,165.474181,28.785771,24.931266,31.328863,102.009834,13.078184,73.268747,...,1232.456175,1609.040485,580000000,NDC,1,NDC,NaN,NaN,NaN,1
NDC1hr-b,94.097834,80.591203,64.648788,175.865826,33.672335,23.812941,32.464085,88.927461,45.302669,86.844990,...,1234.082514,1664.082222,1000000000,NDC,1,NDC,NaN,NaN,NaN,1
NDC1hr-c,92.563834,72.890042,44.378034,158.424808,30.841672,22.895893,18.811935,85.754892,24.501398,75.203266,...,1369.580457,1827.791359,800000000,NDC,1,NDC,NaN,NaN,NaN,1
14CEF1hr-a,64.432483,52.754189,54.063293,128.259588,28.998239,21.943509,47.865737,64.651174,9.697676,49.818086,...,1958.037472,1537.434864,340000000,CEF,1,CEF,NaN,0.25,NaN,1
14CEF1hr-b,67.824743,63.344749,35.632113,127.202811,24.721847,22.006278,32.252596,65.061409,10.607838,45.240048,...,1769.710527,1570.874119,700000000,CEF,1,CEF,NaN,0.25,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34CEF12RIF4hr-b,250.462160,330.726238,38.232498,255.911075,16.701565,23.053869,11.885009,98.896800,12.899583,122.530787,...,962.459468,3231.829820,30000000,CEF+RIF,2,CEF,RIF,0.75,0.50,4
34CEF12RIF4hr-c,226.255231,240.873028,62.899433,230.612976,24.506621,27.978551,31.707580,115.861558,20.648594,98.402187,...,946.507463,2893.918199,32000000,CEF+RIF,2,CEF,RIF,0.75,0.50,4
34CEF34RIF4hr-a,237.790268,299.804077,38.604780,250.452101,36.971501,33.600879,26.309308,140.271276,9.017439,100.513536,...,966.513847,3036.631548,8000000,CEF+RIF,2,CEF,RIF,0.75,0.75,4


In [ ]:
# All functions for data conversion and extraction from specified path
import pandas as pd
import numpy as np
import os
import functools

def read_fcnts_as_df(folder_path):
    """
    Extracts all feature count csvs from a specified folder and stores them as a list of dataframes

    Args:
        folder_path : File path containing Fcnts csv output from pipeline

    Returns: 
        fcnt_df_list : List of Fcnts dataframes
    """
    
    # Filter out the summary files and keep Fcnts from no-drug control (NDC) comparisons
    all_files = os.listdir(folder_path)
    files = [
        f for f in all_files
        if f.endswith(".csv")
        and ".summary" not in f
        and "NDC0hr" in f
            ]

    if len(files) == 0: 
        raise FileNotFoundError(f"No matching feature count files found in {folder_path}")

    # Convert each csv (which is stored as tab-delimited) to dataframe
    files = sorted(files)
    filenames = ["".join([folder_path, "/" , f]) for f in files]
    fcnt_df_list = [pd.read_table(f, sep = ",", header = 0, skiprows = 1) for f in filenames]

    return fcnt_df_list

def sample_name_strip(name):
    """
    Convert a sample file name into an easy to read sample name
    Args:
        name : (ex: "/ExpOut/260107_AV242502_RNASeq_miniHT_SpnT4WT_CEF_CIP/Out/Rep/Bams/T4-wt12CEF12CIP1hr-a.bam")
        
    Returns:
        new_name : (ex: 12CEF12CIP1hr-a)
    """
    # Find index of the last / and remove entire prefix (OG file path)
    samplename_start_idx = name.strip().rfind("/") + 1
    new_name = name[samplename_start_idx:]

    # Find index of . (.bam is at end of sample name) and remove filetag
    filetag_start_idx = new_name.rfind(".")
    new_name = new_name[:filetag_start_idx]

    # Remove "T4-wt"
    new_name = new_name.replace("T4-wt", "")

    return new_name

def fcnts_to_tpms(fcnt_df_list):
    """
    Converts a list of RNA-seq feature count dataframes to a list of TPM dataframes

    Args:
        fcnt_df_list : List of feature count dataframes

    Returns:
        tpm_df_list : List of TPM dataframes

    """
    tpm_df_list = []

    for df in fcnt_df_list:

        # Move gene names to index and keep only length, Fcnts
        df = df.set_index("Geneid")
        df = df.loc[:, [col for col in list(df.columns) if col == "Length" or col.startswith("/")]]
        df = df.apply(lambda column: [int(entry) for entry in column])

        # Convert gene length from b -> kb
        df["Length"] = df["Length"].apply(lambda column: column / 1000)

        # Select Fcnts columns by excluding Length column
        fcnts_cols = [col for col in list(df.columns) if col != "Length"]
        
        # Fcnts / gene length = (Counts per kb)
        df[fcnts_cols] = df[fcnts_cols].apply(lambda column: column / df["Length"])

        # (Counts per kb) * 10^6 / (Total counts/kb) = TPM
        df[fcnts_cols] = df[fcnts_cols].apply(lambda column: column * 10**6 / sum(column))

        # Remove length column 
        df.drop(columns = "Length", inplace = True)
        
        # Apply stripper to each column names
        df = df.rename(columns = lambda column: sample_name_strip(column))
        tpm_df_list.append(df)        

    return tpm_df_list

def bind_tpm_data(tpm_df_list):
    """
    Function to take a list of TPM dataframes, then bind all into 1 dataframe
    Args: 
        tpm_df_list : list of TPM dataframes

    Returns:
        all_tpms [N,G] : dataframe with all TPM values (N samples on row, G genes on column)
    """
    tpm_df_list_uniq = []

    # Column names of 1st DF (all have last 3 cols)
    colnames = list(tpm_df_list[0].columns)

    # Select redundant NDC0hr columns and make new df with just those
    ndc0hr_cols = [col for col in colnames if "NDC0hr" in col]
    ndc0hr_df = tpm_df_list[0][ndc0hr_cols]
    tpm_df_list_uniq.append(ndc0hr_df)

    # Remove NDC0hr columns from all dfs
    for df in tpm_df_list:
        columns = list(df.columns)
        relevant_idx = [col for col in columns if "NDC0hr" not in col]
        stripped_df = df[relevant_idx]
        tpm_df_list_uniq.append(stripped_df)

    all_tpms = reduce(lambda df1, df2 :
                      pd.merge(df1, df2, 
                               left_index = True, 
                               right_index = True, 
                               how = "outer"),
                      tpm_df_list_uniq)

    # Tranpose to get genes on columns
    all_tpms = all_tpms.T

    return all_tpms

def read_cfus(folder_path):
    """
    Function to extract CFUs into a dataframe with conditions on rows, 1 CFU column
    Args:
        folder_path : path to folder containing CFUs (same format as /all_cfus)

    Returns:
        all_cfus [N,1] : df with condition names as index, 1 column of CFUs (N = # samples)
    """

    # Get files
    files = os.listdir(folder_path)
    
    # Select CSV files
    cfu_files = [csv for csv in files if ".csv" in csv]
    cfu_files = sorted(cfu_files)
    cfu_files = ["".join([folder_path, "/", csv]) for csv in files]
    
    # Load each file as a dataframe
    cfu_dfs = [pd.read_table(csv, sep = ",", header = 0) for csv in cfu_files] 

    for i, df in enumerate(cfu_dfs):

        # Melt to get all triplicates stacked
        df = df.melt(id_vars = "Triplicates", var_name = "Condition", value_name = "CFU")

        # Define labels by combining condition + "-" + triplicate label
        df["Labels"] = df["Condition"].str.strip() + "-" + df["Triplicates"].str.strip()

        # Drop unncessary columns
        df = df.drop(columns = ["Condition", "Triplicates"])

        # Move labels to index
        df = df.set_index("Labels")
        cfu_dfs[i] = df
    
    # Concat
    all_cfus = pd.concat(cfu_dfs)

    return all_cfus

# Function to bind TPMs
def bind_all_data(tpm_df, cfu_df):
    """
    Function bind TPM and cfu dfs
    Args:
        tpm_df [N,G] : Dataframe of TPMs, N = # samples, G = # genes, labels on index
        cfu_df [N,1] : Dataframe of CFUs, labels on index

    Returns:
        data_df : [N, G+1] : Dataframe of all TPMs and CFUs as last column
    """
    # Right join so that CFUs exist
    data_df = pd.merge(tpm_df, cfu_df, left_index = True, right_index = True, how = "right")

    return data_df

def data_extract(fcnts_path, cfu_path):
    """
    Function to run entire data extraction pipeline

    Args:
        fcnts_path : Path to folder containing Fcnts output of lab pipeline
        cfu_path   : Path to folder containing CFU csv outputs

    Returns:
        all_data : [N, G+1] : Dataframe of all TPMs and CFUs as last column 
    """
    stored_fcnts = read_fcnts_as_df(fcnts_path)
    stored_tpms  = fcnts_to_tpms(stored_fcnts)
    tpm_df       = bind_tpm_data(stored_tpms)
    cfu_df       = read_cfus(cfu_path)
    all_data     = bind_all_data(tpm_df, cfu_df)
    
    return all_data

## Pre-processing

Check for CFU = 0, which indicates fully dead cells.

In [5]:
# Find idx where CFU = 0, then list the sample ID
zero_idx = np.where(data_df["CFU"] == 0)[0]
print(data_df.index[zero_idx].tolist())

# Check CFU values for the other 2 replicates
rep_names = ["34CEF4hr-a", "34CEF4hr-b"]
cfu_val_check = data_df.loc[rep_names]["CFU"].tolist()
print(f"CFU values for 34CEF4hr-a and 34CEF4hr-b :{cfu_val_check}")

# Remove sample
data_df = data_df[data_df["CFU"] != 0]

['34CEF4hr-c']
CFU values for 34CEF4hr-a and 34CEF4hr-b :[20000000, 20000000]


## Training with stratified split

Implement the traintest split, keeping replicates together and also ensuring even class distribution.

## Training with held-out combination(s)

## Training with sparse combination matrix